In [1]:
from core.llm_router import LLMRouter
from core.memory_store import get_memory_store
from agents.chroma_agent import ChromaAgent
from agents.general_agent import GeneralAgent
from agents.history_agent import HistoryAgent
from agents.web_agent import WebAgent

In [2]:
llm_router = LLMRouter()
memory_store = get_memory_store()

#agents
chroma_agent = ChromaAgent()
web_scrapper_agent = WebAgent()
history_agent = HistoryAgent()
general_agent = GeneralAgent()



🔧 LLM Router initialized with provider: gemini
Loading collections from: /Users/nitinchaube/Studies/IMPS/ALLAboutAI/Project/AdvisorAI/AdvisorAI/AdvisorAI-Web/backend/VectorDB
Trying to load collection AllFacultyResearchInformation with direct client...
Available collections in AllFacultyResearchInformation: ['AllFacultyResearchInformation']
Collection AllFacultyResearchInformation has 373 documents
Successfully loaded collection: AllFacultyResearchInformation
Trying to load collection AllFacultyGeneralInformation with direct client...
Available collections in AllFacultyGeneralInformation: ['AllFacultyGeneralInformation']
Collection AllFacultyGeneralInformation has 373 documents
Successfully loaded collection: AllFacultyGeneralInformation
Trying to load collection AllCourseRelatedData with direct client...
Available collections in AllCourseRelatedData: ['AllCourseRelatedData']
Collection AllCourseRelatedData has 2487 documents
Successfully loaded collection: AllCourseRelatedData
🔧 LLM R

/Users/nitinchaube/Studies/IMPS/ALLAboutAI/Project/AdvisorAI/AdvisorAI/AdvisorAI-Web/backend/chatbot/tools/chroma_tool.py:46: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-chroma package and should be used instead. To use it run `pip install -U :class:`~langchain-chroma` and import as `from :class:`~langchain_chroma import Chroma``.
  chroma_wrapper = Chroma(


In [25]:
def router_node(state):
    query = state.get("query")
    llm = llm_router.get_llm()
    router_prompt = router_prompt = f"""
            You are a routing assistant. Your task is to analyze the user query and output a JSON object with:
            1. A short, descriptive chat name
            2. Step-by-step reasoning
            3. The most relevant tools to use
            4. Confidence level
            5. The single primary tool to use first

            Available Tools:
            1. "general" - General knowledge, definitions, concepts, theories (non-Stevens)
            2. "chroma" - Stevens-specific info (courses, faculty, programs, policies, requirements)
            3. "web" - Current/recent info, availability, updates, contact details
            4. "history" - Conversation context, follow-ups, referencing previous turns

            Decision Rules:
            - Use "general" for: what is, explain, define, how does, why, concept, theory (non-Stevens)
            - Use "chroma" for: courses, faculty, professors, programs, policies (Stevens-specific)
            - Use "web" for: current information, updates, availability, contact details
            - Use "history" for: follow-up questions, references to previous queries

            Think step by step:
            1. What type of information is requested?
            2. Is it Stevens-specific or general knowledge?
            3. Does it require current/recent updates?
            4. Is it dependent on prior context?

            Format:
            Return ONLY a valid JSON object in this format:
            {{
            "chat_name": "short descriptive name",
            "reasoning": "step-by-step reasoning of your decision",
            "tools": ["tool1", "tool2"],
            "confidence": "high/medium/low",
            "primary_tool": "main_tool_to_use_first"
            }}

            Examples:
            - User: "What is machine learning?" 
            → {{"chat_name": "Machine Learning Basics", "reasoning": "...", "tools": ["general"], "confidence": "high", "primary_tool": "general"}}

            - User: "Tell me about Professor Dehnad" 
            → {{"chat_name": "Professor Dehnad Info", "reasoning": "...", "tools": ["chroma", "web"], "confidence": "high", "primary_tool": "chroma"}}

            - User: "Check again" 
            → {{"chat_name": "Follow-up Request", "reasoning": "...", "tools": ["history"], "confidence": "high", "primary_tool": "history"}}

            User Query: "{query}"
    """

    response = llm.invoke([{"role": "user", "content": router_prompt}])
    print(response)
    return response



In [26]:
initial_state = {
    "query": "what courses are offered by Stevens ",
    "user_id": "1222",
    "chat_history": {}
}


In [27]:
resp = router_node(initial_state)

🤖 Getting LLM instance for provider: gemini
content='```json\n{\n  "chat_name": "Stevens Course Offerings",\n  "reasoning": "The user is asking about courses offered by Stevens. This is Stevens-specific information. Therefore, I should use the chroma tool to find information about Stevens courses.",\n  "tools": ["chroma"],\n  "confidence": "high",\n  "primary_tool": "chroma"\n}\n```' additional_kwargs={} response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'model_name': 'gemini-2.0-flash', 'safety_ratings': []} id='run--2a61aded-4ce0-4aaf-9ad2-d19ec09eb4a3-0' usage_metadata={'input_tokens': 578, 'output_tokens': 87, 'total_tokens': 665, 'input_token_details': {'cache_read': 0}}


In [32]:
import json
import re
try:
    tool_decision = json.loads(resp.content.strip())
except json.JSONDecodeError:
    # Try to extract JSON from markdown
    json_match = re.search(r'\{.*\}', resp.content, re.DOTALL)
    if json_match:
        tool_decision = json.loads(json_match.group())
    else:
        raise json.JSONDecodeError("No valid JSON found")

In [35]:
tool_decision

{'chat_name': 'Stevens Course Offerings',
 'reasoning': 'The user is asking about courses offered by Stevens. This is Stevens-specific information. Therefore, I should use the chroma tool to find information about Stevens courses.',
 'tools': ['chroma'],
 'confidence': 'high',
 'primary_tool': 'chroma'}

In [8]:
from langgraph.graph import StateGraph, END
from typing import Dict

def __build_graph()-> StateGraph:
    workflow = StateGraph(dict)
    workflow.add_node("router",router_node)
    workflow.add_node("tools",tools_node)
    workflow.add_node("reason", reason_node)
    workflow.add_node("web_search", web_search_node)
    workflow.add_node("final", final_node)
    workflow.add_node("save", save_node)

    #defining edes
    workflow.set_entry_point("router")
    workflow.add_edge("router", "tools")
    workflow.add_edge("tools","reason")
    workflow.add_conditional_edges(
        "reason",
        should_web_search,
        {
            "web_search":"web_search",
            "final":"final"
        }
    )
    workflow.add_edge("web_search","final")
    workflow.add_edge("final","save")
    workflow.add_edge("save",END)
    graph = workflow.compile()
    graph.draw("graph.png")
    return graph






In [9]:
graph = __build_graph()

NameError: name 'router_node' is not defined

In [ ]:
initial_state = {
    "query": query,
    "user_id": userid,
    "chat_history": chat_history

}
final_state = graph.ainvoke(initial_state)